<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.9-multimodal-rag/notebooks/exercises-6_9.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.9: Multimodal RAG

*Module 6 · Retrieval-Augmented Generation (RAG)*

> Real documents aren’t just text. Annual reports have charts. Manuals have diagrams. Invoices have tables. Standard text RAG ignores all of this. Multimodal RAG extracts, describes, embeds, and retrieves images, tables, and text together — then reasons across all modalities using Gemini’s native vision capabilities.

`Image RAG` · `Table Extraction` · `Gemini Vision` · `Multi-Vector` · `60 min`

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-6.9.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Strategy: Describe Images as Text, Then Embed — `01_image_description.py`
2. Step 2 — Table Extraction — Structured Data into RAG — `02_table_extraction.py`
3. Step 3 — Complete Multimodal RAG Pipeline — `03_multimodal_rag.py`
4. Step 4 — Direct Vision RAG — Send Images to Gemini at Query Time — `04_vision_rag.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy pillow


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Strategy: Describe Images as Text, Then Embed

Gemini Vision describes charts/images. The description becomes a searchable text chunk.

**`01_image_description.py`** · _Block 1 of 4_

IMAGE DESCRIPTION FOR RAG — Vision to text embeddings


In [ ]:
# IMAGE DESCRIPTION FOR RAG — Vision to text embeddings
import google.generativeai as genai
from PIL import Image, ImageDraw
import numpy as np, os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
vision = genai.GenerativeModel("gemini-2.0-flash")

def describe_image_for_rag(image, context=""):
    """Use Gemini Vision to create a searchable text description."""
    prompt = ("Describe this image in detail for a search index. Include:\n"
              "- What type of visual it is (chart, diagram, photo, table)\n"
              "- All data points, numbers, labels, and trends\n"
              "- Key takeaways someone might search for\n")
    if context: prompt += f"Document context: {context}\n"
    resp = vision.generate_content([image, prompt])
    return resp.text.strip()

# Create a synthetic chart image
img = Image.new("RGB", (400, 250), "white")
draw = ImageDraw.Draw(img)
draw.text((20, 10), "Netsetos Revenue (Lakhs)", fill="black")
for i, (q, v) in enumerate([("Q1",45),("Q2",72),("Q3",98),("Q4",135)]):
    h = int(v * 1.2)
    draw.rectangle([40+i*85, 220-h, 40+i*85+60, 220], fill="#0d9488")
    draw.text((55+i*85, 225), q, fill="black")
    draw.text((50+i*85, 220-h-15), str(v), fill="black")

desc = describe_image_for_rag(img, context="Netsetos annual report 2024")
print("Image Description for RAG:\n")
print(f"  {desc[:300]}\n")

# Now embed the description (not the image!)
emb = genai.embed_content(model="models/text-embedding-004", content=desc)["embedding"]
print(f"  Embedding: {len(emb)} dimensions")
print(f"  This text embedding is searchable with queries like:")
print(f"    'What was Q4 revenue?' or 'revenue growth trend'")


> **What just happened?** Gemini Vision described a bar chart: “Revenue chart showing Q1: 45L, Q2: 72L, Q3: 98L, Q4: 135L with consistent growth.” This TEXT description was embedded as a 768-dim vector. Now the query “What was Q4 revenue?” matches this description by cosine similarity. The trick: you embed the DESCRIPTION, not the image pixels. Text embeddings are faster, cheaper, and compatible with your existing vector store.


### Step 2 · Table Extraction — Structured Data into RAG

Extract tables from PDFs/images, convert to markdown, embed as text.

**`02_table_extraction.py`** · _Block 2 of 4_

TABLE EXTRACTION FOR RAG


In [ ]:
# TABLE EXTRACTION FOR RAG
import google.generativeai as genai
from PIL import Image, ImageDraw
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

# Create a synthetic table image
img = Image.new("RGB", (420, 180), "white")
draw = ImageDraw.Draw(img)
rows = ["Course       | Price   | Hours | Rating",
        "GenAI        | 14,999  | 146   | 4.8",
        "Data Science | 12,999  | 120   | 4.7",
        "Cloud GCP    | 11,999  | 110   | 4.6",
        "Full-Stack   |  9,999  |  80   | 4.5"]
for i, row in enumerate(rows):
    draw.text((15, 15+i*30), row, fill="black")

# Extract table as markdown
resp = model.generate_content([img,
    "Extract this table as markdown. Include ALL rows and columns exactly."])
markdown_table = resp.text.strip()

print("Table Extraction:\n")
print(f"  {markdown_table}\n")

# Embed the markdown table as a single chunk
emb = genai.embed_content(model="models/text-embedding-004", content=markdown_table)["embedding"]
print(f"  Embedded as text ({len(emb)} dims)")
print(f"  Searchable: 'which course is cheapest?' or 'highest rated course'")


### Step 3 · Complete Multimodal RAG Pipeline

Ingest text + images + tables into one unified index. Retrieve and reason across all types.

**`03_multimodal_rag.py`** · _Block 3 of 4_

MULTIMODAL RAG — Text + Images + Tables in one index


In [ ]:
# MULTIMODAL RAG — Text + Images + Tables in one index
import google.generativeai as genai
import numpy as np, os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class MultimodalRAG:
    """RAG over text, images, and tables together."""
    def __init__(self):
        self.entries = []  # {type, text, embedding, source, original}
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def _embed(self, text):
        return np.array(genai.embed_content(model="models/text-embedding-004", content=text)["embedding"])

    def add_text(self, text, source="doc"):
        self.entries.append({"type":"text", "text":text, "embedding":self._embed(text), "source":source})

    def add_image(self, image, source="img", context=""):
        """Describe image with vision, embed description."""
        desc = self.model.generate_content([image,
            f"Describe this image in detail for search indexing. Context: {context}"]).text.strip()
        self.entries.append({"type":"image", "text":desc, "embedding":self._embed(desc),
                            "source":source, "original":image})

    def add_table(self, image_or_text, source="table"):
        """Extract table as markdown, embed it."""
        if isinstance(image_or_text, str):
            md = image_or_text
        else:
            md = self.model.generate_content([image_or_text,
                "Extract this table as markdown with ALL data."]).text.strip()
        self.entries.append({"type":"table", "text":md, "embedding":self._embed(md), "source":source})

    def query(self, question, k=3):
        qe = self._embed(question)
        scores = [(i, np.dot(qe,e["embedding"])/(np.linalg.norm(qe)*np.linalg.norm(e["embedding"])))
                  for i,e in enumerate(self.entries)]
        scores.sort(key=lambda x:-x[1])
        top = [self.entries[i] for i,_ in scores[:k]]
        context = "\n\n".join(f"[{e['type'].upper()} from {e['source']}]\n{e['text']}" for e in top)
        resp = self.model.generate_content(
            f"Answer from context ONLY.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:")
        return {"answer": resp.text.strip(),
                "sources": [{"type":e["type"],"source":e["source"]} for e in top]}

# ── Build multimodal index ──
rag = MultimodalRAG()
print("Building Multimodal RAG...\n")

# Text chunks
rag.add_text("Netsetos is an edtech company in Hyderabad offering AI/ML courses.", "about.txt")
rag.add_text("Refund policy: full within 7 days, 50% 8-30 days, none after 30.", "refund.txt")

# Table (as markdown)
rag.add_table("| Course | Price | Hours | Rating |\n|---|---|---|---|\n| GenAI | 14,999 | 146 | 4.8 |\n| Data Science | 12,999 | 120 | 4.7 |\n| Cloud | 11,999 | 110 | 4.6 |", "pricing_table")

# Image (synthetic chart)
from PIL import Image as PILImage, ImageDraw
chart = PILImage.new("RGB", (300,200), "white")
d = ImageDraw.Draw(chart)
d.text((10,5), "Student Growth", fill="black")
for i,(yr,n) in enumerate([("2024",2000),("2025",5000)]):
    h=int(n/40); d.rectangle([40+i*120,180-h,40+i*120+80,180],fill="#0891b2")
    d.text((60+i*120,182),yr,fill="black"); d.text((55+i*120,180-h-15),str(n),fill="black")
rag.add_image(chart, "growth_chart.png", context="Netsetos annual report")

print(f"  Index: {len(rag.entries)} entries ({sum(1 for e in rag.entries if e['type']=='text')} text, "
      f"{sum(1 for e in rag.entries if e['type']=='table')} table, "
      f"{sum(1 for e in rag.entries if e['type']=='image')} image)\n")

# ── Query across modalities ──
for q in ["Which course is the cheapest?",          # table
          "How many students in 2025?",              # image (chart)
          "What is the refund policy?",               # text
          "What is the most expensive course?"]:     # table
    r = rag.query(q)
    print(f"  Q: {q}")
    print(f"  A: {r['answer'][:120]}")
    print(f"  Sources: {r['sources']}\n")


> **What just happened?** “Which course is cheapest?” retrieved the markdown TABLE chunk. “How many students in 2025?” retrieved the IMAGE description (from the chart). “Refund policy?” retrieved the TEXT chunk. All three modalities live in the same vector space as text embeddings. The query doesn’t need to know whether the answer is in text, a table, or a chart — cosine similarity finds it regardless.


### Step 4 · Direct Vision RAG — Send Images to Gemini at Query Time

Alternative: skip text descriptions. Send the actual image to Gemini with retrieved context.

**`04_vision_rag.py`** · _Block 4 of 4_

DIRECT VISION RAG — Send images to Gemini at query time


In [ ]:
# DIRECT VISION RAG — Send images to Gemini at query time
import google.generativeai as genai
import numpy as np, os
from PIL import Image

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class VisionRAG:
    """Store images + text. Send relevant images to Gemini at query time."""
    def __init__(self):
        self.entries = []
        self.model = genai.GenerativeModel("gemini-2.0-flash")

    def _embed(self, text):
        return np.array(genai.embed_content(model="models/text-embedding-004",content=text)["embedding"])

    def add(self, text_desc, source, image=None):
        self.entries.append({"desc":text_desc, "emb":self._embed(text_desc),
                            "source":source, "image":image})

    def query(self, question, k=3):
        qe = self._embed(question)
        scores = sorted([(i,np.dot(qe,e["emb"])/(np.linalg.norm(qe)*np.linalg.norm(e["emb"])))
                         for i,e in enumerate(self.entries)], key=lambda x:-x[1])[:k]
        top = [self.entries[i] for i,_ in scores]

        # Build multimodal prompt: text + images
        parts = [f"Answer from this context ONLY.\n\nQuestion: {question}\n\nContext:\n"]
        for e in top:
            parts.append(f"\n[{e['source']}]: {e['desc']}")
            if e["image"]: parts.append(e["image"])  # send actual image!
        parts.append("\n\nAnswer:")

        resp = self.model.generate_content(parts)
        return {"answer": resp.text.strip(), "sources": [e["source"] for e in top]}

print("Vision RAG: Send images directly to Gemini at query time.\n")
print("  Strategy A (Lesson above): Image -> describe -> embed description -> search text")
print("  Strategy B (This script): Image -> describe for search -> send ACTUAL image at query\n")
print("  Strategy A: faster retrieval, text-only vector store")
print("  Strategy B: more accurate answers (Gemini sees the image), higher cost")
print("  Best practice: Strategy A for retrieval, Strategy B for generation")


> **What just happened?** ColPali eliminates the entire OCR→chunk→embed pipeline with a single vision model forward pass. The key innovation: late interaction via MaxSim allows token-level matching between query words and document patches, catching visual elements (charts, tables, diagrams) that OCR misses entirely. ColQwen2.5 is the current production recommendation. Byaldi makes it 3 lines of code. The tradeoff: multi-vector storage scales fast — use Light-ColPali or binary quantization for large corpora.


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
